In [ ]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# 1. Cargar el dataset final
df = pd.read_csv('dataset_final_colab.csv')

# 1. Asegurar que los datos base existan
# Reutilizamos el DataFrame y el escalador que ya tienes en memoria
try:
    # Verificamos si data_scaled ya existe, si no, hay que generarlo
    print(f"Verificando datos base... Dataset con {df.shape[1]} columnas.")
except NameError:
    print("⚠️ Error: No encontré el DataFrame 'df'. Asegúrate de haber cargado el CSV en la celda anterior.")

# 2. Definir las estaciones adicionales
# Ajusta estos nombres exactos a los de tu archivo CSV
estaciones_objetivo = ['ayacucho', 'caicara','ciudad_bolivar','palua']

# 3. Bucle de entrenamiento
for estacion in estaciones_objetivo:
    if estacion not in df.columns:
        print(f"❌ La estación '{estacion}' no existe en las columnas del dataset. Saltando...")
        continue

    print(f"\n" + "="*50)
    print(f"🛰️  ENTRENANDO ESPECIALISTA: {estacion.upper()}")
    print("="*50)

    # --- A. Creación de Ventanas para esta estación ---
    # Usamos la misma lógica de 60 días que en Ciudad Bolívar
    idx_target = df.columns.get_loc(estacion)

    X_esp = []
    y_esp = []

    # Re-creamos las ventanas para asegurar consistencia
    for i in range(60, len(data_scaled)):
        X_esp.append(data_scaled[i-60:i, :]) # Las 27 variables de entrada
        y_esp.append(data_scaled[i, idx_target]) # El nivel de la estación actual

    X_esp = np.array(X_esp)
    y_esp = np.array(y_esp)

    # Split 80/20 (Temporal)
    split = int(len(X_esp) * 0.8)
    X_train, X_test = X_esp[:split], X_esp[split:]
    y_train, y_test = y_esp[:split], y_esp[split:]

    # --- B. Arquitectura del Modelo (Misma base que el primero) ---
    model = Sequential([
        LSTM(100, return_sequences=True, input_shape=(60, X_esp.shape[2])),
        Dropout(0.2),
        LSTM(50, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # --- C. Callbacks ---
    # EarlyStopping para evitar sobreentrenamiento
    monitor = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5)

    # --- D. Ejecución ---
    print(f"Entrenando {estacion}...")
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_test, y_test),
        callbacks=[monitor, reduce_lr],
        verbose=1 # Cambiado a 1 para que veas el progreso de cada una
    )

    # --- E. Guardar Resultados ---
    model.save(f'modelo_{estacion}.h5')
    print(f"✅ Modelo guardado como: modelo_{estacion}.h5")

print("\n" + "="*50)
print("🎯 ¡PROCESO FINALIZADO CON ÉXITO!")
print("Ya tienes un modelo especialista para cada punto del río.")

In [ ]:
#salida del código anterior 
Verificando datos base... Dataset con 27 columnas.

==================================================
🛰️  ENTRENANDO ESPECIALISTA: AYACUCHO
==================================================
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
Entrenando ayacucho...
Epoch 1/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.0046 - mae: 0.0480 - val_loss: 0.0027 - val_mae: 0.0406 - learning_rate: 0.0010
Epoch 2/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0016 - mae: 0.0309 - val_loss: 0.0036 - val_mae: 0.0481 - learning_rate: 0.0010
Epoch 3/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.0012 - mae: 0.0265 - val_loss: 0.0018 - val_mae: 0.0325 - learning_rate: 0.0010
Epoch 4/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 8.9732e-04 - mae: 0.0233 - val_loss: 0.0021 - val_mae: 0.0348 - learning_rate: 0.0010
Epoch 5/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 6.6847e-04 - mae: 0.0201 - val_loss: 0.0022 - val_mae: 0.0348 - learning_rate: 0.0010
Epoch 6/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 6.0975e-04 - mae: 0.0193 - val_loss: 0.0034 - val_mae: 0.0434 - learning_rate: 0.0010
Epoch 7/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 5.6120e-04 - mae: 0.0185 - val_loss: 0.0031 - val_mae: 0.0414 - learning_rate: 0.0010
Epoch 8/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 4.6929e-04 - mae: 0.0169 - val_loss: 0.0026 - val_mae: 0.0371 - learning_rate: 0.0010
Epoch 9/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 3.1844e-04 - mae: 0.0138 - val_loss: 0.0033 - val_mae: 0.0432 - learning_rate: 2.0000e-04
Epoch 10/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 3.0739e-04 - mae: 0.0136 - val_loss: 0.0027 - val_mae: 0.0385 - learning_rate: 2.0000e-04
Epoch 11/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 2.9272e-04 - mae: 0.0133 - val_loss: 0.0034 - val_mae: 0.0433 - learning_rate: 2.0000e-04
Epoch 12/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 2.8990e-04 - mae: 0.0133 - val_loss: 0.0028 - val_mae: 0.0392 - learning_rate: 2.0000e-04
Epoch 13/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 2.7440e-04 - mae: 0.0129 - val_loss: 0.0024 - val_mae: 0.0359 - learning_rate: 2.0000e-04
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
✅ Modelo guardado como: modelo_ayacucho.h5

==================================================
🛰️  ENTRENANDO ESPECIALISTA: CAICARA
==================================================
Entrenando caicara...
Epoch 1/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - loss: 0.0059 - mae: 0.0509 - val_loss: 0.0025 - val_mae: 0.0408 - learning_rate: 0.0010
Epoch 2/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.0018 - mae: 0.0324 - val_loss: 0.0050 - val_mae: 0.0593 - learning_rate: 0.0010
Epoch 3/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.0013 - mae: 0.0273 - val_loss: 6.8468e-04 - val_mae: 0.0198 - learning_rate: 0.0010
Epoch 4/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 9.8220e-04 - mae: 0.0242 - val_loss: 0.0012 - val_mae: 0.0270 - learning_rate: 0.0010
Epoch 5/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 7.9684e-04 - mae: 0.0218 - val_loss: 0.0018 - val_mae: 0.0338 - learning_rate: 0.0010
Epoch 6/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 6.8333e-04 - mae: 0.0203 - val_loss: 9.5464e-04 - val_mae: 0.0243 - learning_rate: 0.0010
Epoch 7/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 5.3010e-04 - mae: 0.0179 - val_loss: 0.0019 - val_mae: 0.0337 - learning_rate: 0.0010
Epoch 8/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 4.9416e-04 - mae: 0.0172 - val_loss: 0.0021 - val_mae: 0.0366 - learning_rate: 0.0010
Epoch 9/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 3.3517e-04 - mae: 0.0141 - val_loss: 9.5823e-04 - val_mae: 0.0250 - learning_rate: 2.0000e-04
Epoch 10/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 3.0701e-04 - mae: 0.0136 - val_loss: 0.0011 - val_mae: 0.0265 - learning_rate: 2.0000e-04
Epoch 11/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 2.9985e-04 - mae: 0.0134 - val_loss: 0.0014 - val_mae: 0.0291 - learning_rate: 2.0000e-04
Epoch 12/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 2.8385e-04 - mae: 0.0130 - val_loss: 0.0014 - val_mae: 0.0299 - learning_rate: 2.0000e-04
Epoch 13/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 2.7823e-04 - mae: 0.0130 - val_loss: 0.0014 - val_mae: 0.0295 - learning_rate: 2.0000e-04
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
✅ Modelo guardado como: modelo_caicara.h5

==================================================
🛰️  ENTRENANDO ESPECIALISTA: CIUDAD_BOLIVAR
==================================================
Entrenando ciudad_bolivar...
Epoch 1/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - loss: 0.0054 - mae: 0.0482 - val_loss: 0.0026 - val_mae: 0.0420 - learning_rate: 0.0010
Epoch 2/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0016 - mae: 0.0299 - val_loss: 0.0023 - val_mae: 0.0388 - learning_rate: 0.0010
Epoch 3/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 0.0011 - mae: 0.0254 - val_loss: 0.0036 - val_mae: 0.0514 - learning_rate: 0.0010
Epoch 4/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 8.9578e-04 - mae: 0.0228 - val_loss: 0.0014 - val_mae: 0.0293 - learning_rate: 0.0010
Epoch 5/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 7.5378e-04 - mae: 0.0210 - val_loss: 0.0035 - val_mae: 0.0458 - learning_rate: 0.0010
Epoch 6/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 6.1902e-04 - mae: 0.0189 - val_loss: 0.0024 - val_mae: 0.0374 - learning_rate: 0.0010
Epoch 7/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 5.2422e-04 - mae: 0.0176 - val_loss: 0.0040 - val_mae: 0.0476 - learning_rate: 0.0010
Epoch 8/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 4.7442e-04 - mae: 0.0168 - val_loss: 0.0024 - val_mae: 0.0365 - learning_rate: 0.0010
Epoch 9/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - loss: 4.4741e-04 - mae: 0.0163 - val_loss: 0.0016 - val_mae: 0.0320 - learning_rate: 0.0010
Epoch 10/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 3.2572e-04 - mae: 0.0138 - val_loss: 0.0019 - val_mae: 0.0327 - learning_rate: 2.0000e-04
Epoch 11/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - loss: 2.9713e-04 - mae: 0.0132 - val_loss: 0.0027 - val_mae: 0.0390 - learning_rate: 2.0000e-04
Epoch 12/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 2.9401e-04 - mae: 0.0131 - val_loss: 0.0023 - val_mae: 0.0373 - learning_rate: 2.0000e-04
Epoch 13/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 2.9017e-04 - mae: 0.0131 - val_loss: 0.0020 - val_mae: 0.0336 - learning_rate: 2.0000e-04
Epoch 14/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 2.6983e-04 - mae: 0.0126 - val_loss: 0.0020 - val_mae: 0.0342 - learning_rate: 2.0000e-04
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
✅ Modelo guardado como: modelo_ciudad_bolivar.h5

==================================================
🛰️  ENTRENANDO ESPECIALISTA: PALUA
==================================================
Entrenando palua...
Epoch 1/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - loss: 0.0045 - mae: 0.0464 - val_loss: 0.0021 - val_mae: 0.0350 - learning_rate: 0.0010
Epoch 2/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0017 - mae: 0.0308 - val_loss: 0.0036 - val_mae: 0.0444 - learning_rate: 0.0010
Epoch 3/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.0012 - mae: 0.0260 - val_loss: 0.0040 - val_mae: 0.0473 - learning_rate: 0.0010
Epoch 4/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 9.5140e-04 - mae: 0.0235 - val_loss: 0.0029 - val_mae: 0.0406 - learning_rate: 0.0010
Epoch 5/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 7.5827e-04 - mae: 0.0211 - val_loss: 0.0030 - val_mae: 0.0396 - learning_rate: 0.0010
Epoch 6/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 6.9762e-04 - mae: 0.0204 - val_loss: 0.0021 - val_mae: 0.0346 - learning_rate: 0.0010
Epoch 7/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 5.0406e-04 - mae: 0.0173 - val_loss: 0.0025 - val_mae: 0.0369 - learning_rate: 2.0000e-04
Epoch 8/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 4.8585e-04 - mae: 0.0170 - val_loss: 0.0022 - val_mae: 0.0347 - learning_rate: 2.0000e-04
Epoch 9/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 4.7753e-04 - mae: 0.0169 - val_loss: 0.0031 - val_mae: 0.0416 - learning_rate: 2.0000e-04
Epoch 10/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 4.4735e-04 - mae: 0.0163 - val_loss: 0.0020 - val_mae: 0.0330 - learning_rate: 2.0000e-04
Epoch 11/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 4.2490e-04 - mae: 0.0159 - val_loss: 0.0029 - val_mae: 0.0402 - learning_rate: 2.0000e-04
Epoch 12/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 4.2452e-04 - mae: 0.0159 - val_loss: 0.0026 - val_mae: 0.0382 - learning_rate: 2.0000e-04
Epoch 13/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 4.1849e-04 - mae: 0.0158 - val_loss: 0.0019 - val_mae: 0.0321 - learning_rate: 2.0000e-04
Epoch 14/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 3.9327e-04 - mae: 0.0153 - val_loss: 0.0024 - val_mae: 0.0362 - learning_rate: 2.0000e-04
Epoch 15/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 3.9119e-04 - mae: 0.0152 - val_loss: 0.0028 - val_mae: 0.0390 - learning_rate: 2.0000e-04
Epoch 16/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 3.6449e-04 - mae: 0.0146 - val_loss: 0.0022 - val_mae: 0.0346 - learning_rate: 4.0000e-05
Epoch 17/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 3.4720e-04 - mae: 0.0143 - val_loss: 0.0023 - val_mae: 0.0357 - learning_rate: 4.0000e-05
Epoch 18/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 3.5112e-04 - mae: 0.0144 - val_loss: 0.0021 - val_mae: 0.0331 - learning_rate: 4.0000e-05
Epoch 19/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 3.3604e-04 - mae: 0.0141 - val_loss: 0.0021 - val_mae: 0.0335 - learning_rate: 4.0000e-05
Epoch 20/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 3.4228e-04 - mae: 0.0142 - val_loss: 0.0023 - val_mae: 0.0354 - learning_rate: 4.0000e-05
Epoch 21/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 3.3148e-04 - mae: 0.0139 - val_loss: 0.0022 - val_mae: 0.0345 - learning_rate: 8.0000e-06
Epoch 22/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 3.2427e-04 - mae: 0.0139 - val_loss: 0.0023 - val_mae: 0.0349 - learning_rate: 8.0000e-06
Epoch 23/100
401/401 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 3.2517e-04 - mae: 0.0138 - val_loss: 0.0022 - val_mae: 0.0345 - learning_rate: 8.0000e-06
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
✅ Modelo guardado como: modelo_palua.h5

==================================================
🎯 ¡PROCESO FINALIZADO CON ÉXITO!
Ya tienes un modelo especialista para cada punto del río.


In [ ]:
#escaladores 
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

# 1. Cargar tu dataset
df = pd.read_csv('dataset_final_colab.csv')

# 2. Filtrar solo columnas numéricas (quitando 'fecha' si existe)
columnas_numericas = df.select_dtypes(include=[np.number]).columns
df_numeric = df[columnas_numericas]

# --- ESCALADOR GLOBAL (Para las entradas X) ---
# Este se usa para transformar las 27 variables antes de meterlas al modelo
scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_x.fit(df_numeric)
joblib.dump(scaler_x, 'scaler_x_final.pkl')
print("✅ scaler_x_final.pkl creado (27 variables).")

# --- ESCALADORES INDIVIDUALES (Para las salidas Y) ---
# Creamos uno por estación para poder convertir la predicción a metros fácilmente
estaciones = ['ayacucho', 'caicara', 'ciudad_bolivar', 'palua']

for est in estaciones:
    if est in df.columns:
        scaler_y = MinMaxScaler(feature_range=(0, 1))
        # Ajustamos el escalador solo con los datos de esa columna
        scaler_y.fit(df[[est]]) 
        
        nombre_archivo = f'scaler_y_{est}.pkl'
        joblib.dump(scaler_y, nombre_archivo)
        print(f"✅ {nombre_archivo} creado.")

print("\n--- ¡LISTO! ---")
print("Descarga todos los archivos .pkl de la carpeta de la izquierda.")

In [ ]:
#configuración del modelo para predicción en cascada 
import numpy as np
import joblib
from tensorflow.keras.models import load_model

# 1. Cargar "La Flota" (Modelos y Escaladores)
estaciones = ['ayacucho', 'caicara', 'ciudad_bolivar', 'palua']

# Cambiamos load_model para que no intente compilar el optimizador
modelos = {est: load_model(f'modelo_{est}.h5', compile=False) for est in estaciones}
scaler_x = joblib.load('scaler_x_final.pkl')
scalers_y = {est: joblib.load(f'scaler_y_{est}.pkl') for est in estaciones}

def prediccion_en_cascada(datos_entrada_60dias):
    """
    datos_entrada_60dias: Matriz de (60, 27) con los datos actuales
    """
    resultados = {}
    
    # Preparamos el buffer de entrada (X_input)
    # Lo convertimos a formato (1, 60, 27) para Keras
    current_x = np.expand_dims(datos_entrada_60dias, axis=0)
    
    for est in estaciones:
        # A. Predecir (esto nos da un valor entre 0 y 1)
        pred_scaled = modelos[est].predict(current_x, verbose=0)
        
        # B. Guardar resultado convertido a metros reales
        pred_metros = scalers_y[est].inverse_transform(pred_scaled)
        resultados[est] = pred_metros[0][0]
        
        # C. EFECTO CASCADA: 
        # Si no es la última estación, actualizamos el input para la siguiente
        # Insertamos nuestra predicción en la columna correspondiente para el "mañana" imaginario
        # Nota: Esto es útil si quieres predecir a varios días vista (Multi-step)
        
    return resultados

print("✅ Sistema de cascada configurado y listo.")

In [ ]:
import matplotlib.pyplot as plt

# 1. Preparar los datos de prueba
# Tomamos los últimos datos del dataset para validar
split = int(len(data_scaled) * 0.8)
test_data = data_scaled[split:]

# Creamos las ventanas de 60 días para el test
X_test_all = []
for i in range(60, len(test_data)):
    X_test_all.append(test_data[i-60:i, :])
X_test_all = np.array(X_test_all)

# 2. Ejecutar Predicciones
print("Calculando predicciones en cascada...")
fig, axes = plt.subplots(4, 1, figsize=(12, 18))
plt.subplots_adjust(hspace=0.4)

for i, est in enumerate(estaciones):
    # Predicción del modelo
    preds_scaled = modelos[est].predict(X_test_all, verbose=0)
    preds = scalers_y[est].inverse_transform(preds_scaled)
    
    # Valores reales (des-escalados)
    idx_target = df.columns.get_loc(est)
    reales_scaled = test_data[60:, idx_target].reshape(-1, 1)
    reales = scalers_y[est].inverse_transform(reales_scaled)
    
    # Graficar
    axes[i].plot(reales, label='Real (Histórico)', color='blue', alpha=0.6)
    axes[i].plot(preds, label='Predicción IA', color='red', linestyle='--')
    axes[i].set_title(f'Estación: {est.upper()}')
    axes[i].set_ylabel('Nivel (m)')
    axes[i].legend()
    axes[i].grid(True)

plt.show()

In [ ]:
#simulación en cascada (9 años )
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from tqdm.notebook import tqdm

# 1. Configuración inicial y preparación de datos
orden_cascada = ['ayacucho', 'caicara', 'ciudad_bolivar', 'palua']
indices_estaciones = {est: df.columns.get_loc(est) for est in orden_cascada}

split = int(len(data_scaled) * 0.8)
test_data_clean = data_scaled[split:].copy()
predicciones_cascada = {est: [] for est in orden_cascada}

total_pasos = len(test_data_clean) - 60
print(f"🚀 Iniciando Simulación en Cascada para {total_pasos} días...")

# 2. Bucle de ejecución con Barra de Progreso
for i in tqdm(range(60, len(test_data_clean)), desc="Procesando flujo del río"):
    
    # Extraemos la ventana de 60 días una sola vez por paso de tiempo
    ventana_actual = test_data_clean[i-60:i, :].copy()
    
    for est in orden_cascada:
        # Preparar entrada para la red LSTM (batch, timesteps, features)
        X_input = ventana_actual.reshape(1, 60, ventana_actual.shape[1])
        
        # Predicción puntual
        pred_point = modelos[est].predict(X_input, verbose=0)
        val_pred = pred_point[0, 0]
        
        # Almacenar predicción
        predicciones_cascada[est].append(val_pred)
        
        # --- EFECTO CASCADA ---
        # Inyectamos la predicción recién hecha en la ventana actual
        # para que la siguiente estación río abajo la use como dato de entrada
        idx_col = indices_estaciones[est]
        ventana_actual[-1, idx_col] = val_pred

print("\n✅ Simulación completada. Generando visualización...")

# 3. Visualización y Métricas
fig, axes = plt.subplots(4, 1, figsize=(12, 24))
plt.subplots_adjust(hspace=0.5)

for i, est in enumerate(orden_cascada):
    # Obtener valores reales y predicciones (des-escalados a metros)
    idx_col = indices_estaciones[est]
    y_real_scaled = test_data_clean[60:, idx_col].reshape(-1, 1)
    
    y_real = scalers_y[est].inverse_transform(y_real_scaled)
    y_pred = scalers_y[est].inverse_transform(np.array(predicciones_cascada[est]).reshape(-1, 1))
    
    # Calcular métrica de similitud R²
    score = r2_score(y_real, y_pred) * 100
    
    # Eje X convertido a meses (30.41 días promedio)
    eje_x_meses = np.arange(len(y_real)) / 30.41
    
    # Graficar
    axes[i].plot(eje_x_meses, y_real, label='Nivel Real (Histórico)', color='blue', alpha=0.5, linewidth=1.5)
    axes[i].plot(eje_x_meses, y_pred, label=f'IA Cascada (R²: {score:.2f}%)', color='green', linestyle='--', linewidth=1.5)
    
    # Estética de la gráfica
    axes[i].set_title(f'SISTEMA EN CASCADA: {est.upper()}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Nivel (Metros)')
    axes[i].set_xlabel('Tiempo transcurrido (Meses)')
    axes[i].legend(loc='upper right')
    axes[i].grid(True, linestyle=':', alpha=0.6)
    
    # Cuadro de texto con la precisión
    axes[i].text(0.02, 0.92, f'Similitud R²: {score:.2f}%', 
                 transform=axes[i].transAxes, fontsize=11, 
                 bbox=dict(facecolor='white', alpha=0.7))

plt.show()

In [ ]:
#simulación en cascada 2 años 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from tqdm.notebook import tqdm

# 1. Configuración y preparación de datos
orden_cascada = ['ayacucho', 'caicara', 'ciudad_bolivar', 'palua']
indices_estaciones = {est: df.columns.get_loc(est) for est in orden_cascada}

# Mantenemos el split original para asegurar que los datos no fueron usados en entrenamiento
split = int(len(data_scaled) * 0.8)
test_data_total = data_scaled[split:].copy()

# --- CAMBIO CLAVE: Reducción a los últimos 2 años (730 días) ---
dias_a_testear = 730 
# Nos aseguramos de tener 60 días adicionales atrás para la primera ventana
inicio_bucle = len(test_data_total) - dias_a_testear 

if inicio_bucle < 60:
    inicio_bucle = 60 # Seguridad por si el dataset es más pequeño

predicciones_cascada = {est: [] for est in orden_cascada}
total_pasos = len(test_data_total) - inicio_bucle

print(f"⏱️ Iniciando Simulación Rápida (Últimos 2 años): {total_pasos} días...")

# 2. Bucle de ejecución con Barra de Progreso
for i in tqdm(range(inicio_bucle, len(test_data_total)), desc="Simulando Cascada"):
    
    ventana_actual = test_data_total[i-60:i, :].copy()
    
    for est in orden_cascada:
        X_input = ventana_actual.reshape(1, 60, ventana_actual.shape[1])
        
        # Predicción
        pred_point = modelos[est].predict(X_input, verbose=0)
        val_pred = pred_point[0, 0]
        
        predicciones_cascada[est].append(val_pred)
        
        # Inyección en cascada
        idx_col = indices_estaciones[est]
        ventana_actual[-1, idx_col] = val_pred

print("\n✅ Simulación reducida completada.")

# 3. Visualización y Métricas
fig, axes = plt.subplots(4, 1, figsize=(12, 22))
plt.subplots_adjust(hspace=0.5)

for i, est in enumerate(orden_cascada):
    idx_col = indices_estaciones[est]
    
    # Extraemos solo los valores reales que corresponden al periodo testeado
    y_real_scaled = test_data_total[inicio_bucle:, idx_col].reshape(-1, 1)
    
    y_real = scalers_y[est].inverse_transform(y_real_scaled)
    y_pred = scalers_y[est].inverse_transform(np.array(predicciones_cascada[est]).reshape(-1, 1))
    
    score = r2_score(y_real, y_pred) * 100
    eje_x_meses = np.arange(len(y_real)) / 30.41
    
    axes[i].plot(eje_x_meses, y_real, label='Nivel Real', color='blue', alpha=0.6)
    axes[i].plot(eje_x_meses, y_pred, label=f'IA Cascada (R²: {score:.2f}%)', color='green', linestyle='--')
    
    axes[i].set_title(f'TEST REDUCIDO (2 AÑOS): {est.upper()}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Metros')
    axes[i].set_xlabel('Meses')
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)
    
    # Marcador de precisión
    axes[i].text(0.02, 0.9, f'Similitud: {score:.2f}%', transform=axes[i].transAxes, 
                 bbox=dict(facecolor='white', alpha=0.8))

plt.show()

In [ ]:
#simulación en cascada con predicción a 10 días en el futoro 
# ==========================================
# 3. VISUALIZACIÓN CON ESCALADORES REALES
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
plt.subplots_adjust(hspace=0.4, wspace=0.3)
axes = axes.flatten()

print("\n📈 Traduciendo a metros reales y graficando...")

for i, est in enumerate(orden_cascada):
    # 1. Obtener las predicciones escaladas (0 a 1)
    preds_scaled = np.array(resultados_scaled[est]).reshape(-1, 1)
    
    # 2. USAR TU ESCALADOR ESPECÍFICO
    # Aquí usamos el objeto que guardaste antes. 
    # Asegúrate de que 'scalers_y[est]' sea el MinMaxScaler entrenado con los metros reales.
    try:
        metros = scalers_y[est].inverse_transform(preds_scaled)
    except NameError:
        print(f"❌ Error: No se encontró el objeto 'scalers_y'.")
        metros = preds_scaled # Fallback para no romper el código
    
    # --- CONFIGURACIÓN DE LA GRÁFICA ---
    dias_eje = np.arange(1, dias_a_predecir + 1)
    y_min, y_max = np.min(metros), np.max(metros)
    rango = y_max - y_min if y_max != y_min else 1.0
    
    axes[i].plot(dias_eje, metros, marker='o', markersize=8, color='darkcyan', linewidth=2.5)
    
    # Ajuste de límites para que las etiquetas queden dentro
    axes[i].set_ylim(y_min - (rango*0.2), y_max + (rango*0.5))
    axes[i].set_title(f'PRONÓSTICO 10 DÍAS: {est.upper()}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Nivel en Metros (Real)')
    axes[i].set_xticks(dias_eje)
    axes[i].grid(True, linestyle='--', alpha=0.5)
    
    # Etiquetas de texto con el valor real en metros
    for x, y in zip(dias_eje, metros):
        axes[i].annotate(f'{y[0]:.2f}m', 
                         xy=(x, y), 
                         xytext=(0, 12), 
                         textcoords='offset points', 
                         ha='center', 
                         fontweight='bold',
                         bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="darkcyan", alpha=0.8))

plt.suptitle('TENDENCIA HIDROLÓGICA EN METROS REALES', fontsize=18, fontweight='bold', y=0.96)
plt.show()